In [14]:
import pandas as pd 
import tqdm
from google import genai
from dotenv import load_dotenv 
from pinecone import Pinecone 
import os 

In [15]:
# retrieve necessary keys from .env 
load_dotenv() 
pinecone_key = os.getenv("Pinecone_key") 
gemini_key = os.getenv("Gemini_key") 

In [30]:
# hyperparameters 
pinecone_top_k=2

In [17]:
# load clients 
client = genai.Client(api_key=gemini_key) 
pc = Pinecone(api_key=pinecone_key) 

In [41]:
# retrieve top k results from pinecone output to create context 
def retrieve_topk_text(results, top_k): 
    '''
    return array of top_k texts 
    ''' 
    return [hit["fields"]["chunk_text"].replace('\xad', '') for hit in (results['result']['hits'][:top_k])]

In [19]:
df = pd.read_csv("/Users/xiang/Desktop/Advising_Bot/data/Pinecone - Sheet1.csv")

In [20]:
df['_id'] = df['_id'].astype(str) 

In [21]:
df.head()

,_id,chunk_text
0,vec_1,Maintenance of Public Order: The University wi...
1,vec_2,Campus Safety: The Advisory Committee on Campu...
2,vec_3,Student Responsibility: Students are responsib...
3,vec_4,Bias-Related Crime Prevention: The Stony Brook...
4,vec_5,Student Educational Records: The Federal Famil...


In [22]:
records = df.to_dict("records")

In [23]:
index_name = "advising-bot"
if not pc.has_index(index_name): 
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

In [24]:
dense_index = pc.Index(index_name)

/opt/anaconda3/envs/newEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
namespace = "bulletin"
dense_index.upsert_records(namespace, records)

UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Fri, 13 Mar 2026 22:19:17 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '3', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '584', 'x-pinecone-response-duration-ms': '586', 'server': 'envoy'}})

### Does flipped order and same order create the same embedding? 


In [50]:
query= "Which course is Psychology 101?"

results = dense_index.search(
    namespace=namespace,
    query={
        "top_k": pinecone_top_k,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']:
        print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")

id: vec_8 | score: 0.19  | text: Degree Requirements:Have completed all prerequisites with a grade of C or higher. (Pass/No Credit grades are not acceptable to meet prerequisites.) For transfer students, official transfer credit evaluations must have been completed.
id: vec_7 | score: 0.19  | text: Computer Science Electives: Six additional upper-division technical CSE courses, each of which must carry at least three credits. Courses used to satisfy the required advanced courses requirement may not be used to satisfy the computer science electives requirement. Technical electives do not include teaching practica (CSE 475), the first part of the senior honors project (CSE 495), and courses designated as non-technical in the course description (such as CSE 301). Students may only use 3 credits from the following courses to satisfy one upper-division technical elective for the CSE major requirements: CSE 487, CSE 496, VIP 395, VIP 396, VIP 495, VIP 496. 


In [51]:
pinecone_results = retrieve_topk_text(results, pinecone_top_k)

#### Build Gemini API as LLM backbone 

In [40]:
gemini_model = "gemini-3-flash-preview" 
system_role= "You are a strict, factual AI assistant answering college students' answers. " \
            "Questions are answered using ONLY provided context. If the context does not contain the answer," \
            "you should answer: 'I do not have this information available'. " \
            "These context are provided to you right begore the user question under the <context> tags"

In [53]:
# all the context
formatted_context = " \n- ".join(pinecone_results)

In [54]:
prompt = f""" {system_role} 
Below is the context:  

<context>
- {formatted_context} 
</context>
Question: {query}
"""

In [55]:
prompt

" You are a strict, factual AI assistant answering college students' answers. Questions are answered using ONLY provided context. If the context does not contain the answer,you should answer: 'I do not have this information available'. These context are provided to you right begore the user question under the <context> tags \nBelow is the context:  \n\n<context>\n- Degree Requirements:Have completed all prerequisites with a grade of C or higher. (Pass/No Credit grades are not acceptable to meet prerequisites.) For transfer students, official transfer credit evaluations must have been completed. \n- Computer Science Electives: Six additional upper-division technical CSE courses, each of which must carry at least three credits. Courses used to satisfy the required advanced courses requirement may not be used to satisfy the computer science electives requirement. Technical electives do not include teaching practica (CSE 475), the first part of the senior honors project (CSE 495), and cour

In [56]:
response = client.models.generate_content(model=gemini_model, contents=prompt)

In [57]:
print(response.text)

I do not have this information available.
